# Fase 2h: CT ensemble de modelos mixed-context

Este notebook combina las probabilidades de los dos mejores modelos CT 2D. No entrena un modelo nuevo: selecciona el peso del ensemble y el threshold usando validacion, y evalua una sola vez en test.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.config import config
from src.data.segmentation import (
    build_ct_segmentation_dataframe,
    split_segmentation_dataframe,
    SegmentationDataset,
    SegmentationPairTransform,
)
from src.models.segmentation import build_segmentation_model
from src.training.segmentation_experiment import (
    compute_segmentation_metrics_from_probs,
    get_device,
    summarize_segmentation_results,
)

config.NUM_WORKERS = 0
device = get_device()
print(f'Device: {device}')

## 2. Modelos base

In [ ]:
ENSEMBLE_EXPERIMENTS = {
    'mixed50_patch160_pos20': {
        'experiment': 'ct_attention_unet_mixed50_patch160_pos80_tversky_pos20_thr_segmentation',
        'base_features': 16,
        'in_channels': 1,
        'ct_context_slices': False,
    },
    'mixed30_patch192_pos10': {
        'experiment': 'ct_attention_unet_mixed30_patch192_pos70_tversky_pos10_thr080_segmentation',
        'base_features': 16,
        'in_channels': 1,
        'ct_context_slices': False,
    },
}

ENSEMBLE_NAME = 'ct_attention_unet_ensemble_mixed50_mixed30_segmentation'
summary_path = PROJECT_ROOT / 'results' / 'segmentation' / 'ct' / f'{ENSEMBLE_NAME}_full_summary.json'
summary_path

## 3. Datos y splits

In [ ]:
ct_seg_df = build_ct_segmentation_dataframe(
    config.CT_DIR,
    config.CT_DIR / 'processed_segmentation_slices',
    target_size=config.CT_IMAGE_SIZE,
    positive_mask_only=True,
    overwrite=False,
)

train_df, val_df, test_df = split_segmentation_dataframe(
    ct_seg_df,
    random_seed=config.RANDOM_SEED,
    group_col='study_id',
)

print({'train': len(train_df), 'val': len(val_df), 'test': len(test_df)})
print({'train_studies': train_df.study_id.nunique(), 'val_studies': val_df.study_id.nunique(), 'test_studies': test_df.study_id.nunique()})

## 4. Utilidades de inferencia

In [ ]:
def load_attention_unet_checkpoint(experiment, in_channels=1, base_features=16):
    model = build_segmentation_model(
        architecture='attention_unet',
        in_channels=in_channels,
        out_channels=1,
        base_features=base_features,
    )
    checkpoint_path = config.MODELS_DIR / 'segmentation' / 'ct' / f'{experiment}_full.pt'
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    state_dict = checkpoint['model_state_dict'] if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint else checkpoint
    model.load_state_dict(state_dict)
    return model.to(device).eval()


def collect_probabilities(df, model_specs):
    outputs = {}
    masks = None
    for name, spec in model_specs.items():
        model = load_attention_unet_checkpoint(
            spec['experiment'],
            in_channels=spec['in_channels'],
            base_features=spec['base_features'],
        )
        transform = SegmentationPairTransform(
            image_size=config.CT_IMAGE_SIZE,
            in_channels=spec['in_channels'],
            is_train=False,
        )
        dataset = SegmentationDataset(
            df,
            transform=transform,
            in_channels=spec['in_channels'],
            ct_context_slices=spec['ct_context_slices'],
        )
        loader = DataLoader(dataset, batch_size=8, shuffle=False, num_workers=0)
        probs = []
        current_masks = []
        with torch.no_grad():
            for images, batch_masks in loader:
                logits = model(images.to(device))
                probs.append(torch.sigmoid(logits).cpu())
                current_masks.append(batch_masks.cpu())
        outputs[name] = torch.cat(probs)
        if masks is None:
            masks = torch.cat(current_masks)
    return outputs, masks

## 5. Seleccion de peso y threshold en validacion

In [ ]:
val_probs, val_masks = collect_probabilities(val_df, ENSEMBLE_EXPERIMENTS)

weights = np.arange(0.0, 1.01, 0.05)
thresholds = np.arange(0.60, 0.96, 0.05)
rows = []

for old_weight in weights:
    ensemble_probs = (
        old_weight * val_probs['mixed50_patch160_pos20']
        + (1.0 - old_weight) * val_probs['mixed30_patch192_pos10']
    )
    for threshold in thresholds:
        metrics = compute_segmentation_metrics_from_probs(ensemble_probs, val_masks, threshold=float(threshold))
        rows.append({
            'old_weight': float(old_weight),
            'new_weight': float(1.0 - old_weight),
            'threshold': float(threshold),
            **metrics,
        })

val_grid = pd.DataFrame(rows).sort_values(['dice', 'iou'], ascending=False).reset_index(drop=True)
best = val_grid.iloc[0].to_dict()
print(best)
val_grid.head(10)

## 6. Evaluacion en test con la seleccion de validacion

In [ ]:
test_probs, test_masks = collect_probabilities(test_df, ENSEMBLE_EXPERIMENTS)
ensemble_test_probs = (
    best['old_weight'] * test_probs['mixed50_patch160_pos20']
    + best['new_weight'] * test_probs['mixed30_patch192_pos10']
)

test_metrics = compute_segmentation_metrics_from_probs(
    ensemble_test_probs,
    test_masks,
    threshold=best['threshold'],
)

summary = {
    'experiment': ENSEMBLE_NAME,
    'dataset': 'ct',
    'run_mode': 'full',
    'architecture': 'attention_unet_ensemble',
    'variant_name': 'ensemble_mixed50_patch160_pos20_mixed30_patch192_pos10_val_tuned',
    'loss_name': 'probability_average',
    'dice': test_metrics['dice'],
    'iou': test_metrics['iou'],
    'pixel_accuracy': test_metrics['pixel_accuracy'],
    'threshold': best['threshold'],
    'split_sizes': {'train': len(train_df), 'val': len(val_df), 'test': len(test_df)},
    'hyperparameters': {
        'old_model': ENSEMBLE_EXPERIMENTS['mixed50_patch160_pos20']['experiment'],
        'new_model': ENSEMBLE_EXPERIMENTS['mixed30_patch192_pos10']['experiment'],
        'old_weight': best['old_weight'],
        'new_weight': best['new_weight'],
        'selection_split': 'val',
        'selection_metric': 'dice',
        'threshold_grid': [float(x) for x in thresholds],
        'weight_grid': [float(x) for x in weights],
    },
}
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.write_text(json.dumps(summary, indent=2))

print(summary_path)
summary

## 7. Comparacion CT

In [ ]:
summary_df = summarize_segmentation_results(sorted((PROJECT_ROOT / 'results' / 'segmentation' / 'ct').glob('*_summary.json')))
cols = ['experiment', 'variant_name', 'loss_name', 'dice', 'iou', 'pixel_accuracy', 'threshold', 'hyperparameters']
summary_df[cols].sort_values('dice', ascending=False).reset_index(drop=True)